# Ardook grant fit on real grants

Tests models on Timothy's 72 grants (rows 38–109 of `data/real_grants.csv`), labeled for Ardook. Each grant's text is `X`; Timothy's `Fit` rating, mapped to High / Moderate / Low, is `y`.

| `Fit` | Label | Grants |
|---|---|---:|
| Excellent, Strong | High | 11 |
| Moderate, Weak-Moderate | Moderate | 24 |
| Weak | Low | 37 |

**Models never see** "Why It Fits", "Key Concern", "Recommended Action" or `Fit`. Geography is ignored (team decision).

Three kinds of methods:
1. **Trained on synthetic grants** (generated from the team-reviewed `phrase_pools.csv`), then tested on all 72 real grants.
2. **Zero-shot NLI**: no training; a fixed rule over yes/no statements.
3. **Trained on the real labels** with 5-fold cross-validation, so each grant is predicted by a model that never saw its label.


In [1]:
%pip -q install "sentence-transformers>=3.0" "transformers>=4.46" datasets accelerate setfit

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Hide Apple MPS first, before any library checks for it: it crashes these models (NSException)
# or mixes CPU and MPS tensors. CUDA or CPU is used instead.
import torch
_orig_mps = torch.backends.mps.is_available
def _no_mps():
    return False
_no_mps.__wrapped__ = getattr(_orig_mps, "__wrapped__", _orig_mps)
torch.backends.mps.is_available = _no_mps

import gc, os, random, re, shutil, time, warnings
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
LABELS = ["Low", "Moderate", "High"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
print("device:", DEVICE)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1111 21:08:26.596000 88787 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


device: cpu


## Real grants

In [3]:
raw = pd.read_csv("../data/real_grants.csv", dtype=str, keep_default_na=False)
raw["row"] = pd.to_numeric(raw.iloc[:, 0], errors="coerce")
real = raw[raw.row >= 38].copy()          # Timothy's rows, rated for Ardook
FIT_MAP = {"Excellent": "High", "Strong": "High", "Moderate": "Moderate", "Weak-Moderate": "Moderate", "Weak": "Low"}
real["fit_raw"] = real.Fit.str.strip()
real["label"] = real.fit_raw.map(FIT_MAP)
assert real.label.notna().all(), real.fit_raw[real.label.isna()].unique()
clean = lambda s: re.sub(r"\s+", " ", str(s)).strip()
real["text"] = [f"{clean(p)}. {clean(t)}. {clean(a)}. {clean(c)}" for p, t, a, c in
                zip(real.Program, real.Type, real["Amount / Benefit"], real["Core Requirements"])]
real = real[["row", "Program", "fit_raw", "label", "text"]].reset_index(drop=True)
real["row"] = real.row.astype(int)
y_real = real.label.map(LABEL2ID).values
assert not real.text.str.contains("Ardook", case=False).any()  # no labeler reasoning in the input
print(len(real), real.label.value_counts().to_dict())
real.head(3)

72 {'Low': 37, 'Moderate': 24, 'High': 11}


,row,Program,fit_raw,label,text
0,38,Women Founders Network,Moderate,Moderate,"Women Founders Network. Grant. $25,000-$55,000..."
1,39,Founderland Forward Grant,Moderate,Moderate,"Founderland Forward Grant. Grant. €10,000.00. ..."
2,40,Risk and Protective Factors of Family Health a...,Weak-Moderate,Moderate,Risk and Protective Factors of Family Health a...


## Synthetic grants

Theme and entity decide the label (any `bad` → Low, both `good` → High, otherwise Moderate). Locations appear in about 60% of grants but never change the label. Half of all themes get "for women" added whatever their rating, so "women" alone never signals a good fit. Validation uses held-out phrases, templates and funder names (every 4th item), so it shows overfitting.

In [4]:
# Synthetic grant generator v2: theme + entity decide the label; geography is never part of it.

POOLS = pd.read_csv("phrase_pools.csv")  # reviewed by the team; edit there, not here
def pool(attr, rating):
    return POOLS[(POOLS.attribute == attr) & (POOLS.rating == rating)].phrase.tolist()
THEMES = {r: pool("theme", r) for r in ["good", "partial", "bad"]}
ENTITIES = {r: pool("entity", r) for r in ["good", "partial", "bad"]}
GEOS = pool("geography", "neutral")

FUNDER_PREFIX = ["Northbridge", "Silver Linden", "Harbor Light", "Open Field", "Cedar & Stone", "Kestrel",
                 "Brightwater", "Lumen", "Tidewell", "Granite Peak", "Willowmere", "Amber Coast",
                 "Juniper", "Starling", "Riverbend", "Copperleaf", "Meadowgate", "Bluehaven", "Oakline", "Fairmount"]
FUNDER_SUFFIX = ["Foundation", "Fund", "Trust", "Initiative", "Network", "Collective"]
TYPES = ["Grant", "Microgrant", "Prize", "Award", "Fellowship", "Program", "Challenge", "Seed Fund"]
AMOUNTS = ["$2,000", "$5,000", "$10,000", "$25,000", "Up to $50,000", "$5,000-$30,000", "€10,000",
           "Up to €75,000", "Varies by application", "Not listed"]
# Added to any theme regardless of its rating, so "women" alone never signals a good fit.
AUDIENCE = ["with a focus on women", "for women and girls", "led by or benefiting women", "for female founders"]

# {req} is the requirement list in shuffled order. Every 4th item of each list is validation-only.
TEMPLATES = [
    "{program}. {type}. {amount}. {req_semi}.",
    "{program} ({type}, {amount}). Requirements: {req_semi}.",
    "{program}. {type}. {amount}. {req_sent}",
    "{type}: {program}. Award: {amount}. Eligibility and focus: {req_comma}.",
    "{program}. {type}. {amount}. Supports {theme}. {entity_cap}. {geo_sent}",
    "{program}. {type}. {amount}. {entity_cap}; {theme}; {geo_semi}",
    "{program} offers {amount} for {theme}. {entity_cap}. {geo_sent}",
    "{program}. {type}. {amount}. Funding priorities: {theme}. Who can apply: {entity}. {geo_sent}",
]

def split_pool(items, split):
    return [p for i, p in enumerate(items) if (i % 4 == 3) == (split == "val")]

def fit_label(theme, entity):
    if "bad" in (theme, entity):
        return "Low"
    if (theme, entity) == ("good", "good"):
        return "High"
    return "Moderate"

def make_grant(rng, split, theme_r, entity_r):
    pick = lambda items: rng.choice(split_pool(items, split))
    theme = pick(THEMES[theme_r])
    if rng.random() < 0.5:
        theme = f"{theme} {rng.choice(AUDIENCE)}"
    entity = pick(ENTITIES[entity_r])
    geo = pick(GEOS) if rng.random() < 0.6 else ""
    reqs = [theme, entity] + ([f"open to {geo}"] if geo else [])
    rng.shuffle(reqs)
    cap = lambda s: s[0].upper() + s[1:]
    text = pick(TEMPLATES).format(
        program=f"{pick(FUNDER_PREFIX)} {rng.choice(FUNDER_SUFFIX)} {rng.choice(['', 'Women ', 'Impact ', 'Community '])}{rng.choice(TYPES)}",
        type=rng.choice(TYPES), amount=rng.choice(AMOUNTS),
        req_semi="; ".join(cap(r) for r in reqs), req_comma=", ".join(reqs),
        req_sent=" ".join(cap(r) + "." for r in reqs),
        theme=theme, entity=entity, entity_cap=cap(entity),
        geo_sent=f"Open to {geo}." if geo else "", geo_semi=f"open to {geo}." if geo else "")
    return re.sub(r"\s+", " ", text).strip()

def generate(n_per_class, split, seed):
    rng = random.Random(seed)
    counts, rows, seen = {l: 0 for l in LABELS}, [], set()
    while min(counts.values()) < n_per_class:
        t, e = rng.choice(["good", "partial", "bad"]), rng.choice(["good", "partial", "bad"])
        label = fit_label(t, e)
        if counts[label] >= n_per_class:
            continue
        text = make_grant(rng, split, t, e)
        if text in seen:
            continue
        seen.add(text); counts[label] += 1
        rows.append({"text": text, "label": label, "theme": t, "entity": e})
    df = pd.DataFrame(rows)
    df["label_id"] = df.label.map(LABEL2ID)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


In [5]:
train_df, val_df = generate(800, "train", 11), generate(100, "val", 12)
y_tr, y_va = train_df.label_id.values, val_df.label_id.values
print("train", train_df.label.value_counts().to_dict(), "| val", val_df.label.value_counts().to_dict())
train_df.text.head(3).tolist()

train {'Low': 800, 'Moderate': 800, 'High': 800} | val {'Moderate': 100, 'Low': 100, 'High': 100}


['Kestrel Foundation Women Challenge. Challenge. $25,000. Only government agencies may apply. Decent work and economic growth (SDG 8).',
 'Tidewell Initiative Women Challenge (Award, Not listed). Requirements: Economic independence for caregivers of disabled family members; Only registered nonprofit or charitable organizations; Open to organizations in Israel.',
 'Oakline Network Women Program. Award. Up to $50,000. Programs serving women and children. Only research teams with laboratory facilities.']

## Scoring helpers

In [6]:
def f1_interval(y, p, n=1000):
    rng = np.random.default_rng(SEED)
    idx = rng.integers(0, len(y), (n, len(y)))
    return np.percentile([f1_score(y[i], p[i], average="macro", zero_division=0) for i in idx], [2.5, 97.5])

results, predictions = [], {}

def record(name, kind, p, extra=None):
    p = np.asarray(p)
    pr, rc, f, _ = precision_recall_fscore_support(y_real, p, average="macro", zero_division=0, labels=[0, 1, 2])
    hp, hr, _, _ = precision_recall_fscore_support(y_real, p, zero_division=0, labels=[2])
    lo, hi = f1_interval(y_real, p)
    row = {"method": name, "kind": kind, "accuracy": accuracy_score(y_real, p), "precision": pr, "recall": rc,
           "f1": f, "f1_95ci": f"{lo:.3f}-{hi:.3f}", "high_recall": hr[0], "high_precision": hp[0]}
    row.update(extra or {})
    results.append(row); predictions[name] = p
    print(f"{name:48s} acc={row['accuracy']:.3f} F1={f:.3f} [{lo:.3f}-{hi:.3f}] High R={hr[0]:.3f} P={hp[0]:.3f}")
    print("   confusion (rows true Low/Mod/High):", confusion_matrix(y_real, p, labels=[0, 1, 2]).tolist())

record("0 always Low", "floor", np.zeros(len(real), dtype=int))

0 always Low                                     acc=0.514 F1=0.226 [0.191-0.256] High R=0.000 P=0.000
   confusion (rows true Low/Mod/High): [[37, 0, 0], [24, 0, 0], [11, 0, 0]]


## 1. bge-small criterion scores + logistic regression, trained on synthetic

In [7]:
BGE = "BAAI/bge-small-en-v1.5"
BGE_INSTRUCTION = "Represent this sentence for searching relevant passages: "
CRITERIA = [
    "jobs for mothers of children with disabilities", "disability inclusion", "support for caregivers",
    "women's economic empowerment", "home-based flexible work", "social enterprise creating decent jobs",
    "livelihoods for single mothers",
    "small business growth", "women entrepreneurs", "artificial intelligence and technology research",
    "scientific research", "education and scholarships", "health and medical care", "arts and culture",
    "politics and advocacy",
    "open to any type of organization", "only nonprofit organizations can apply", "limits on company age or revenue",
    "requires women's ownership", "requires a university or research institution", "requires deep technology or patents",
    "only individual students can apply", "requires founders from a specific identity group",
    "an equity investment rather than a grant",
]
bge = SentenceTransformer(BGE, device=DEVICE)
enc = lambda xs: bge.encode(list(xs), batch_size=64, normalize_embeddings=True)
crit = enc([BGE_INSTRUCTION + c for c in CRITERIA])
EMB_REAL = enc(real.text)
CRIT = {"train": enc(train_df.text) @ crit.T, "val": enc(val_df.text) @ crit.T, "real": EMB_REAL @ crit.T}
best = None
for C in [0.1, 1, 10, 100]:   # chosen on synthetic validation
    clf = LogisticRegression(C=C, max_iter=3000).fit(CRIT["train"], y_tr)
    v = f1_score(y_va, clf.predict(CRIT["val"]), average="macro")
    if best is None or v > best[0]:
        best = (v, clf)
record("1 bge criteria + LogReg (synthetic)", "trained on synthetic", best[1].predict(CRIT["real"]), {"synthetic_val_f1": best[0]})
del bge; gc.collect()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8125.25it/s]

1 bge criteria + LogReg (synthetic)              acc=0.500 F1=0.423 [0.311-0.536] High R=0.364 P=0.222
   confusion (rows true Low/Mod/High): [[26, 6, 5], [9, 6, 9], [4, 3, 4]]


3816

## 2. DistilBERT fine-tuned on synthetic

In [8]:
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

NAME = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForSequenceClassification.from_pretrained(NAME, num_labels=3, id2label=dict(enumerate(LABELS)), label2id=LABEL2ID)
def to_ds(texts, labels):
    return Dataset.from_dict({"text": list(texts), "labels": list(labels)}).map(
        lambda b: tok(b["text"], truncation=True, max_length=128), batched=True, remove_columns=["text"])
out_dir = "/tmp/ardook_distilbert_real"
args = TrainingArguments(output_dir=out_dir, num_train_epochs=3, learning_rate=5e-5, weight_decay=0.01,
                         per_device_train_batch_size=16, per_device_eval_batch_size=64,
                         eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
                         load_best_model_at_end=True, metric_for_best_model="f1_macro",
                         use_cpu=DEVICE == "cpu", logging_steps=50, report_to="none", seed=SEED)
trainer = Trainer(model=model, args=args, train_dataset=to_ds(train_df.text, y_tr), eval_dataset=to_ds(val_df.text, y_va),
                  processing_class=tok, data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=lambda ev: {"f1_macro": f1_score(ev[1], ev[0].argmax(-1), average="macro")})
trainer.train()
val_f1 = f1_score(y_va, trainer.predict(to_ds(val_df.text, y_va)).predictions.argmax(-1), average="macro")
DISTIL_LOGITS = trainer.predict(to_ds(real.text, y_real)).predictions
assert np.isfinite(DISTIL_LOGITS).all()
record("2 DistilBERT (synthetic)", "trained on synthetic", DISTIL_LOGITS.argmax(-1), {"synthetic_val_f1": val_f1})
SYNTH_DIR = "/tmp/ardook_distilbert_synth"   # kept for method 8
trainer.save_model(SYNTH_DIR)
del trainer, model; gc.collect(); shutil.rmtree(out_dir, ignore_errors=True)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7336.80it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Map:   0%|          | 0/2400 [00:00<?, ? examples/s]


Map:  83%|████████▎ | 2000/2400 [00:00<00:00, 12778.42 examples/s]


Map: 100%|██████████| 2400/2400 [00:00<00:00, 12872.36 examples/s]

... (16 outputs left out)

2 DistilBERT (synthetic)                         acc=0.625 F1=0.532 [0.409-0.666] High R=0.182 P=0.500
   confusion (rows true Low/Mod/High): [[24, 13, 0], [3, 19, 2], [3, 6, 2]]



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.43s/it]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.43s/it]

## 3. Zero-shot NLI

`MoritzLaurer/deberta-v3-base-zeroshot-v2.0` rates how likely each statement is true for each grant. Fixed rule, 0.5 threshold: Low if Ardook can't apply, or the theme is off-mission and doesn't match Ardook's; High if the theme matches and no unknown-fact requirement applies; otherwise Moderate.

- **3**: the original 8 statements, written before any results. Several are long "or" lists, which the model almost never judges true.
- **3b** (*revised after seeing results*): 21 short single-topic statements; also checked on 150 synthetic validation grants.

In [9]:
from transformers import pipeline

nli = pipeline("zero-shot-classification", model="MoritzLaurer/deberta-v3-base-zeroshot-v2.0", device=0 if DEVICE == "cuda" else -1)

def nli_scores(texts, hyps):
    P = np.zeros((len(texts), len(hyps)))
    for i, t in enumerate(texts):
        out = nli(t, candidate_labels=hyps, hypothesis_template="{}", multi_label=True)
        d = dict(zip(out["labels"], out["scores"]))
        P[i] = [d[h] for h in hyps]
    assert np.isfinite(P).all()
    return P

def nli_rule(P, hyps, groups):
    true = lambda g: P[:, [hyps.index(h) for h in groups[g]]].max(axis=1) > 0.5
    tg, tb, eb, ep = true("theme_good"), true("theme_bad"), true("entity_bad"), true("entity_partial")
    return np.where(eb | (tb & ~tg), 0, np.where(tg & ~ep, 2, 1))

# 3: original statements (written before any results)
GROUPS_V1 = {
    "theme_good": ["This grant supports employment or income for women, mothers, or caregivers.",
                   "This grant supports people with disabilities or their families.",
                   "This grant supports women's economic empowerment or social enterprises that create jobs."],
    "theme_bad": ["This grant is mainly for education, scholarships, research, health, arts, or politics."],
    "entity_bad": ["Only students, individuals, universities, schools, or researchers can apply."],
    "entity_partial": ["Only nonprofit organizations can apply.",
                       "Applicants must belong to a specific racial, ethnic, or LGBTQ+ group.",
                       "The funding is an equity investment in a company."],
}
# 3b: short single-topic statements (revised after seeing results)
GROUPS_V2 = {
    "theme_good": ["This grant creates jobs or income for women.", "This grant supports mothers.",
                   "This grant supports caregivers.", "This grant supports people with disabilities.",
                   "This grant supports women's economic empowerment.", "This grant supports social enterprises."],
    "theme_bad": ["This grant is about education.", "This is a scholarship for students.",
                  "This grant funds scientific or medical research.", "This grant is about health care.",
                  "This grant is about arts and culture.", "This grant is about politics or government.",
                  "This grant is about violence prevention.", "This grant is about technology or artificial intelligence."],
    "entity_bad": ["Only students can apply.", "Only universities or researchers can apply.", "Only individuals can apply."],
    "entity_partial": ["Only nonprofit organizations can apply.", "Applicants must be people of color.",
                       "Applicants must be LGBTQ+.", "The funding is an equity investment."],
}
t = time.time()
H1 = [h for g in GROUPS_V1.values() for h in g]
H2 = [h for g in GROUPS_V2.values() for h in g]
NLI_REAL_V1 = nli_scores(real.text, H1)
NLI_REAL_V2 = nli_scores(real.text, H2)
val150 = val_df.groupby("label", group_keys=False).apply(lambda d: d.sample(50, random_state=SEED))
v2_val_f1 = f1_score(val150.label_id, nli_rule(nli_scores(val150.text, H2), H2, GROUPS_V2), average="macro")
record("3 NLI rule, 8 statements", "zero-shot", nli_rule(NLI_REAL_V1, H1, GROUPS_V1))
record("3b NLI rule, 21 statements", "zero-shot, revised after seeing results", nli_rule(NLI_REAL_V2, H2, GROUPS_V2),
       {"synthetic_val_f1": v2_val_f1})
print(f"NLI took {time.time()-t:.0f}s")
del nli; gc.collect()



Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 6902.24it/s]

3 NLI rule, 8 statements                         acc=0.250 F1=0.184 [0.104-0.271] High R=0.182 P=0.154
   confusion (rows true Low/Mod/High): [[0, 34, 3], [0, 16, 8], [0, 9, 2]]


3b NLI rule, 21 statements                       acc=0.361 F1=0.346 [0.232-0.455] High R=0.364 P=0.174
   confusion (rows true Low/Mod/High): [[10, 19, 8], [1, 12, 11], [0, 7, 4]]
NLI took 1373s


0

## Trained on the real labels (5-fold cross-validation)

Stratified 5-fold, repeated with 5 shuffles; settings fixed in advance (StandardScaler + LogisticRegression, C=1, balanced class weights). The table records the first shuffle; the mean and spread over all shuffles are in `f1_mean` / `f1_sd`.

In [10]:
def cv_runs(fit_predict, repeats):
    runs = []
    for r in range(repeats):
        p = np.zeros(len(y_real), dtype=int)
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED + r).split(real.text, y_real):
            p[te] = fit_predict(tr, te)
        runs.append(p)
    return runs

def record_cv(name, runs, kind="trained on real (CV)"):
    f1s = [f1_score(y_real, p, average="macro") for p in runs]
    record(name, kind, runs[0], {"f1_mean": np.mean(f1s), "f1_sd": np.std(f1s), "shuffles": len(runs)})
    print(f"   over {len(runs)} shuffles: F1 {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")

FEATURES = {"4 bge embeddings + LogReg (real, CV)": EMB_REAL,
            "5 bge criteria + LogReg (real, CV)": CRIT["real"],
            "7 NLI scores, 8 statements + LogReg (real, CV)": NLI_REAL_V1,
            "7b NLI scores, 21 statements + LogReg (real, CV)": NLI_REAL_V2,
            "9 DistilBERT outputs + NLI 8 statements + LogReg (real, CV)": np.hstack([DISTIL_LOGITS, NLI_REAL_V1])}
for name, X in FEATURES.items():
    fp = lambda tr, te, X=X: make_pipeline(StandardScaler(), LogisticRegression(C=1, class_weight="balanced", max_iter=3000)).fit(X[tr], y_real[tr]).predict(X[te])
    record_cv(name, cv_runs(fp, 5), "trained on real (CV)" + (", revised after seeing results" if name.startswith("7b") else ""))

4 bge embeddings + LogReg (real, CV)             acc=0.417 F1=0.340 [0.237-0.444] High R=0.182 P=0.143
   confusion (rows true Low/Mod/High): [[22, 8, 7], [13, 6, 5], [3, 6, 2]]
   over 5 shuffles: F1 0.328 ± 0.019


5 bge criteria + LogReg (real, CV)               acc=0.417 F1=0.322 [0.237-0.407] High R=0.091 P=0.059
   confusion (rows true Low/Mod/High): [[24, 8, 5], [8, 5, 11], [3, 7, 1]]
   over 5 shuffles: F1 0.319 ± 0.030


7 NLI scores, 8 statements + LogReg (real, CV)   acc=0.514 F1=0.482 [0.368-0.587] High R=0.545 P=0.261
   confusion (rows true Low/Mod/High): [[21, 10, 6], [3, 10, 11], [1, 4, 6]]
   over 5 shuffles: F1 0.468 ± 0.017


7b NLI scores, 21 statements + LogReg (real, CV) acc=0.458 F1=0.420 [0.311-0.525] High R=0.364 P=0.154
   confusion (rows true Low/Mod/High): [[20, 6, 11], [4, 9, 11], [1, 6, 4]]
   over 5 shuffles: F1 0.419 ± 0.031


9 DistilBERT outputs + NLI 8 statements + LogReg (real, CV) acc=0.542 F1=0.497 [0.372-0.602] High R=0.545 P=0.286
   confusion (rows true Low/Mod/High): [[24, 8, 5], [5, 9, 10], [2, 3, 6]]
   over 5 shuffles: F1 0.485 ± 0.029


## 6. SetFit: fine-tunes bge-small's weights on the real labels (CV)

In [11]:
from setfit import SetFitModel, Trainer as SetFitTrainer, TrainingArguments as SetFitArgs

SETFIT_SHUFFLES = 1   # each shuffle trains 5 models; raise on a GPU

def setfit_fp(tr, te):
    m = SetFitModel.from_pretrained(BGE, device=DEVICE)
    SetFitTrainer(model=m, args=SetFitArgs(batch_size=16, num_epochs=1, num_iterations=20, seed=SEED, report_to="none"),
                  train_dataset=Dataset.from_dict({"text": real.text[tr].tolist(), "label": y_real[tr].tolist()})).train()
    return np.asarray(m.predict(real.text[te].tolist()))

t = time.time()
record_cv("6 SetFit bge-small (real, CV)", cv_runs(setfit_fp, SETFIT_SHUFFLES))
print(f"SetFit took {time.time()-t:.0f}s")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7172.46it/s]

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.



Map:   0%|          | 0/57 [00:00<?, ? examples/s]


Map: 100%|██████████| 57/57 [00:00<00:00, 23397.47 examples/s]

***** Running training *****


... (56 outputs left out)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.21it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.21it/s]

6 SetFit bge-small (real, CV)                    acc=0.528 F1=0.360 [0.278-0.432] High R=0.000 P=0.000
   confusion (rows true Low/Mod/High): [[28, 9, 0], [14, 10, 0], [9, 2, 0]]
   over 1 shuffles: F1 0.360 ± 0.000
SetFit took 658s


## 8. DistilBERT trained on synthetic, then fine-tuned on each real training fold (CV)

In [12]:
t = time.time()
p8 = np.zeros(len(y_real), dtype=int)
for tr_idx, te_idx in StratifiedKFold(5, shuffle=True, random_state=SEED).split(real.text, y_real):
    m = AutoModelForSequenceClassification.from_pretrained(SYNTH_DIR)
    fa = TrainingArguments(output_dir="/tmp/ardook_fold", num_train_epochs=5, learning_rate=2e-5, per_device_train_batch_size=8,
                           save_strategy="no", use_cpu=DEVICE == "cpu", report_to="none", seed=SEED, logging_steps=1000)
    ft = Trainer(model=m, args=fa, train_dataset=to_ds(real.text[tr_idx], y_real[tr_idx]),
                 processing_class=tok, data_collator=DataCollatorWithPadding(tok))
    ft.train()
    p8[te_idx] = ft.predict(to_ds(real.text[te_idx], y_real[te_idx])).predictions.argmax(-1)
    del ft, m; gc.collect(); shutil.rmtree("/tmp/ardook_fold", ignore_errors=True)
record("8 DistilBERT synthetic -> real (CV)", "synthetic then real (CV)", p8)
print(f"took {time.time()-t:.0f}s")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6028.96it/s]


Map:   0%|          | 0/57 [00:00<?, ? examples/s]


Map: 100%|██████████| 57/57 [00:00<00:00, 2547.28 examples/s]

Step,Training Loss



Map:   0%|          | 0/15 [00:00<?, ? examples/s]

... (26 outputs left out)

Step,Training Loss



Map:   0%|          | 0/14 [00:00<?, ? examples/s]


Map: 100%|██████████| 14/14 [00:00<00:00, 4179.08 examples/s]

8 DistilBERT synthetic -> real (CV)              acc=0.583 F1=0.457 [0.351-0.583] High R=0.091 P=0.250
   confusion (rows true Low/Mod/High): [[27, 10, 0], [7, 14, 3], [4, 6, 1]]
took 59s


## Results

In [13]:
res = pd.DataFrame(results)
os.makedirs("../results", exist_ok=True)
res.to_csv("../results/real_grant_results.csv", index=False)
display(Markdown(res.drop(columns=["kind"]).round(3).to_markdown(index=False)))

| method                                                      |   accuracy |   precision |   recall |    f1 | f1_95ci     |   high_recall |   high_precision |   synthetic_val_f1 |   f1_mean |   f1_sd |   shuffles |
|:------------------------------------------------------------|-----------:|------------:|---------:|------:|:------------|--------------:|-----------------:|-------------------:|----------:|--------:|-----------:|
| 0 always Low                                                |      0.514 |       0.171 |    0.333 | 0.226 | 0.191-0.256 |         0     |            0     |            nan     |   nan     | nan     |        nan |
| 1 bge criteria + LogReg (synthetic)                         |      0.5   |       0.43  |    0.439 | 0.423 | 0.311-0.536 |         0.364 |            0.222 |              0.468 |   nan     | nan     |        nan |
| 2 DistilBERT (synthetic)                                    |      0.625 |       0.6   |    0.541 | 0.532 | 0.409-0.666 |         0.182 |            0.5   |              0.66  |   nan     | nan     |        nan |
| 3 NLI rule, 8 statements                                    |      0.25  |       0.142 |    0.283 | 0.184 | 0.104-0.271 |         0.182 |            0.154 |            nan     |   nan     | nan     |        nan |
| 3b NLI rule, 21 statements                                  |      0.361 |       0.466 |    0.378 | 0.346 | 0.232-0.455 |         0.364 |            0.174 |              0.412 |   nan     | nan     |        nan |
| 4 bge embeddings + LogReg (real, CV)                        |      0.417 |       0.341 |    0.342 | 0.34  | 0.237-0.444 |         0.182 |            0.143 |            nan     |     0.328 |   0.019 |          5 |
| 5 bge criteria + LogReg (real, CV)                          |      0.417 |       0.332 |    0.316 | 0.322 | 0.237-0.407 |         0.091 |            0.059 |            nan     |     0.319 |   0.03  |          5 |
| 7 NLI scores, 8 statements + LogReg (real, CV)              |      0.514 |       0.506 |    0.51  | 0.482 | 0.368-0.587 |         0.545 |            0.261 |            nan     |     0.468 |   0.017 |          5 |
| 7b NLI scores, 21 statements + LogReg (real, CV)            |      0.458 |       0.461 |    0.426 | 0.42  | 0.311-0.525 |         0.364 |            0.154 |            nan     |     0.419 |   0.031 |          5 |
| 9 DistilBERT outputs + NLI 8 statements + LogReg (real, CV) |      0.542 |       0.503 |    0.523 | 0.497 | 0.372-0.602 |         0.545 |            0.286 |            nan     |     0.485 |   0.029 |          5 |
| 6 SetFit bge-small (real, CV)                               |      0.528 |       0.342 |    0.391 | 0.36  | 0.278-0.432 |         0     |            0     |            nan     |     0.36  |   0     |          1 |
| 8 DistilBERT synthetic -> real (CV)                         |      0.583 |       0.476 |    0.468 | 0.457 | 0.351-0.583 |         0.091 |            0.25  |            nan     |   nan     | nan     |        nan |

In [ ]:
pred_df = real[["row", "Program", "fit_raw", "label"]].copy()
for name, p in predictions.items():
    pred_df[name.split(" ")[0]] = [LABELS[i] for i in p]
pred_df.to_csv("../results/real_grant_predictions.csv", index=False)
pred_df.head(10)

,row,Program,fit_raw,label,0,1,2,3,3b,4,5,7,7b,9,6,8
0,38,Women Founders Network,Moderate,Moderate,Low,Moderate,Moderate,High,High,High,Moderate,Moderate,Moderate,Moderate,Moderate,Moderate
1,39,Founderland Forward Grant,Moderate,Moderate,Low,Moderate,Moderate,High,High,Moderate,Moderate,Moderate,Low,Moderate,Moderate,Moderate
2,40,Risk and Protective Factors of Family Health a...,Weak-Moderate,Moderate,Low,High,Moderate,Moderate,Moderate,Low,Low,High,Low,Low,Low,Moderate
3,41,European Prize for Woman Innovators(EIC),Moderate,Moderate,Low,Low,Moderate,High,High,Moderate,Low,Moderate,Moderate,Moderate,Moderate,Moderate
4,42,Women's Peace and Humanitarian Fund,Strong,High,Low,Moderate,Moderate,Moderate,Moderate,Moderate,Moderate,High,Moderate,High,Moderate,Moderate
5,43,Women's Peace and Humanitarian Fund,Moderate,Moderate,Low,Low,Moderate,Moderate,High,Low,Low,High,Moderate,High,Moderate,Low
6,44,Global Fund for Women,Moderate,Moderate,Low,High,Moderate,High,Moderate,Moderate,High,Moderate,High,Moderate,Moderate,High
7,45,One Meadows Foundation,Moderate,Moderate,Low,High,Moderate,Moderate,High,Moderate,High,High,High,High,Moderate,Moderate
8,46,Tory Burch Foundation Fellows Program,Moderate,Moderate,Low,Moderate,Moderate,High,High,Moderate,High,High,High,High,Moderate,Moderate
9,47,Altrusa International Foundation\n,Weak,Low,Low,Low,Moderate,Moderate,Moderate,Moderate,Moderate,High,High,High,Moderate,Moderate
